# 上下文并行组：
- 它的核心思想是将一个超长输入序列在“序列长度”维度上切分，分散到多个 GPU 上并行处理，从而突破单卡显存限制，支持百万级 Token 的上下文窗口。
- 在训练超长序列的大语言模型时，单卡显存无法容纳整个序列的 KV Cache（键值缓存）。上下文并行（CP） 是一种将长序列切分到多个 GPU 上并行计算的技术（类似于 Ring Attention 或 Block-wise Attention）。

- 它的主要任务是：<font color='red'>根据当前的全局进程排名（rank），找到属于同一个“上下文并行组”的其他进程，</font>并建立底层的通信通道（Process Group）。此外，它还处理了更复杂的“分层上下文并行”情况。
  
## 场景设定

假设你正在训练一个大模型，需要处理一本 200页的小说（这就是“长序列”）。
- 总长度：假设这本小说有 16,000 个 Token。
- 硬件限制：你只有 4 张 GPU 卡（GPU 0, 1, 2, 3）。
- 单卡瓶颈：每张卡的显存只能装下 4,000 个 Token 的数据。如果直接把整本书塞给一张卡，显存瞬间爆炸（OOM）。

| GPU 编号 | 负责的内容（Token 范围） | 本地显存占用 |
| :--- | :--- | :--- |
| GPU 0 | 第 1 - 4,000 个 Token (第 1-50 页) | ✅ 正常 |
| GPU 1 | 第 4,001 - 8,000 个 Token (第 51-100 页) | ✅ 正常 |
| GPU 2 | 第 8,001 - 12,000 个 Token (第 101-150 页) | ✅ 正常 |
| GPU 3 | 第 12,001 - 16,000 个 Token (第 151-200 页) | ✅ 正常 |



In [ ]:
   '''
    
    '''
    # Build the context-parallel groups.
    
    '''定义全局变量来存储当前进程所属的 CP 通信组对象和组内所有进程的 ID 列表。'''
    global _CONTEXT_PARALLEL_GROUP
    global _CONTEXT_PARALLEL_GLOBAL_RANKS
    
    '''断言：确保这个初始化函数只被调用一次。如果 _CONTEXT_PARALLEL_GROUP 已经有值了，说明之前已经初始化过了，再次调用会报错，防止重复创建资源。'''
    assert _CONTEXT_PARALLEL_GROUP is None, 'context parallel group is already initialized'
    
    '''
    遍历并创建标准 CP 组:
    
    获取分组列表：decoder_rank_generator.get_ranks('cp') 会根据你配置的并行策略（TP, PP, DP, CP），
                返回一个包含多个列表的迭代器。每个列表代表一个 CP 组内的所有 global_rank。
    
    例如：如果有 8 张卡，配置为 TP=2, CP=2, DP=2。那么可能会生成类似 [[0, 2], [1, 3], [4, 6], [5, 7]] 这样的组。

    '''
    for ranks in decoder_rank_generator.get_ranks('cp'):
        '''
        创建通信组 (create_group)：
        - 使用 PyTorch 的 torch.distributed.new_group (通常封装在 create_group 中) 来建立通信链路。
        - pg_options：配置底层 NCCL 通信库的参数（如超时时间、带宽设置等），专门针对 "cp" 流量进行优化。
        '''
        group = create_group(
            ranks,
            timeout=timeout,
            pg_options=get_nccl_options("cp", nccl_comm_cfgs),
            group_desc="CONTEXT_PARALLEL_GROUP",
        )
        '''
        认领归属 (if rank in ranks)：
        - 循环会遍历所有的组，但当前进程（self.rank）只属于其中一个组。
        - 一旦发现当前的 rank 在当前的 ranks 列表中，就将这个 group 对象赋值给全局变量 _CONTEXT_PARALLEL_GROUP。
        - 同时记录下这个组里有哪些人（_CONTEXT_PARALLEL_GLOBAL_RANKS），方便后续调试或查看拓扑
        
        '''
        if rank in ranks:
            _CONTEXT_PARALLEL_GROUP = group
            _CONTEXT_PARALLEL_GLOBAL_RANKS = ranks
            
        '''
        处理分层上下文并行:
        
        '''
        if hierarchical_context_parallel_sizes:
            assert np.prod(hierarchical_context_parallel_sizes) == context_parallel_size
            global _HIERARCHICAL_CONTEXT_PARALLEL_GROUPS
            
            
            hierarchical_groups, _ = create_hierarchical_groups(
                rank,
                ranks,
                hierarchical_context_parallel_sizes,
                create_gloo_process_groups=False,
                pg_options=get_nccl_options("hcp", nccl_comm_cfgs),
                timeout=timeout,
                group_desc="CONTEXT_PARALLEL_GROUP",
            )
            if rank in ranks:
                _HIERARCHICAL_CONTEXT_PARALLEL_GROUPS = hierarchical_groups



## 处理分层上下文并行:
- if hierarchical_context_parallel_sizes:

这部分是为了应对超大规模集群或复杂网络拓扑（如 NVLink + InfiniBand 混合架构）的高级功能。
1. 什么是分层 CP？
- 标准的 CP 假设所有参与并行的 GPU 之间的通信速度是一样的。但在实际集群中，<font color='red'>同一节点内的 GPU 通信（NVLink）比跨节点通信（IB/RoCE）快得多。</font>
- 为了优化性能，可以将 CP 组进一步<font color='red'>拆分为“内部组”和“外部组”</font>。例如，先在同一节点内做一部分聚合，再跨节点做聚合。

2. 逻辑检查：
- np.prod(...) == context_parallel_size：<font color='red'>确保分层大小的乘积等于总的 CP 大小</font>。例如总 CP=4，分层可能是 [2, 2]（<font color='red'>2个节点，每节点2卡</font>）。

3. 创建分层组：
- <font color='red'>调用 create_hierarchical_groups 创建嵌套的通信组结构。</font>
- 同样地，只有当 rank 属于当前正在处理的 CP 组时，才将这些分层组保存到全局变量中。

## 3. 举例说明

假设我们有 8 张 GPU (Rank 0-7)，配置如下：
- Tensor Parallel (TP): 2
- Context Parallel (CP): 2
- Data Parallel (DP): 2

###### 场景模拟：
1. Generator 生成组：
get_ranks('cp') 可能会生成以下 4 个组（具体顺序取决于 Rank 映射策略，这里假设一种常见情况）：
- 组 A: [0, 2] (<font color='red'>Rank 0 和 Rank 2 互为 CP 伙伴</font>)
- 组 B: [1, 3]
- 组 C: [4, 6]
- 组 D: [5, 7]

2. 循环执行：
- 当代码处理到 组 A [0, 2] 时：
  - <font color='red'>系统调用底层 API 建立 Rank 0 和 Rank 2 之间的专属高速公路。</font>
  - Rank 0 检查：if 0 in [0, 2] -> True。
    - Rank 0 记录：“我的 CP 组建立了，我的搭档是 Rank 2”。
  - Rank 1 检查：if 1 in [0, 2] -> False。Rank 1 忽略这次循环。
  - ...以此类推。
  
3. 最终结果：
- <font color='red'>Rank 0 拥有了与 Rank 2 通信的能力。</font>
- <font color='green'>在后续的 Attention 计算中，Rank 0 就可以通过 _CONTEXT_PARALLEL_GROUP 向 Rank 2 发送它的 Key/Value 块，或者接收 Rank 2 的数据，从而完成长序列的拼接计算。</font>

##  分层通信组 create_hierarchical_groups
- 调用 create_hierarchical_groups 创建嵌套的通信组结构。
- https://github.com/NVIDIA/Megatron-LM/blob/core_v0.17.0/megatron/core/parallel_state.py#L360

- 它的核心目的是：为了适应现代数据中心的物理拓扑（如 NVLink、InfiniBand），将一组 GPU 按照特定的层级结构切分成多个子组，以便在不同层级上选择最高效的通信方式。

### 1. 什么是“分层通信组”？
在传统的并行训练中，我们通常只建立一个扁平的通信组（例如所有参与张量并行的 GPU 都在一个组里）。但在高性能集群中，GPU 之间的连接速度差异巨大：
- NVLink/Shared Memory：同一节点内的 GPU 通信极快。
- InfiniBand/RoCE：跨节点的 GPU 通信较慢。


###### 💻 结合 16 张 GPU 的代码案例拆解

在你的代码示例中，hierarchical_group_sizes = [2, 2, 4]，总共有 16 张 GPU（g0 到 g15）。
- <font color='red'>我们来看看以 g0 的视角，它所在的三个层级分别是谁，以及为什么要这么分。</font>
#### 第 1 层 (Level 0)：最底层的“亲密搭档” (Size = 2)
- 代码逻辑：将 16 个人按顺序每 2 个切一刀。
- 分组结果：[g0, g1], [g2, g3], [g4, g5] ...
- <font color='red'>g0 的搭档：g1</font>
- 物理意义：<font color='green'>同一台服务器内，走 NVLink 的极速通道。</font>
  - 在计算时，g0 和 g1 的数据交换几乎不消耗跨机带宽，速度最快。它们先完成最基础的局部数据同步。
  
#### 第 2 层 (Level 1)：中间层的“部门协作” (Size = 2)
- 代码逻辑：<font color='red'>每隔 2 个人取一个（步长为 2）。</font>
- 分组结果：[g0, g2], [g1, g3], [g4, g6] ...
- g0 的搭档：g2
- 物理意义：同一台服务器内，但可能跨越了不同的 NVSwitch 或 PCIe 根复合体。
  - <font color='green'>g0 和 g2 的通信速度比 g0-g1 稍慢，但依然在本机内，不需要走外网。这一层负责把两个“亲密小组”的数据聚合起来。</font>
  
#### 第 3 层 (Level 2)：最上层的“跨区汇总” (Size = 4)
- 代码逻辑：<font color='red'>每隔 4 个人取一个（步长为 4）。</font>
- 分组结果：[g0, g4, g8, g12], [g1, g5, g9, g13] ...
- g0 的搭档：g4, g8, g12
- 物理意义：<font color='red'>跨越不同的服务器节点，走 InfiniBand (IB) 或 RoCE 网络。</font>
  - <font color='green'>这是最慢的一层通信！g0、g4、g8、g12 可能分别插在 4 台不同的物理服务器上。</font>
  - <font color='blue'>因为前两层已经在本机内把数据“预聚合”过了，所以这一层只需要传输少量汇总后的核心数据，避免了 16 张卡同时挤占宝贵的跨机带宽。</font>
  
## 代码逻辑深度解析
- 函数的核心在于如何根据输入的 hierarchical_group_sizes 将原始的 ranks 列表重新排列并切分。这里使用了 einops 库来进行高维数组的重排。

###### rearranged_ranks = einops.rearrange原理解析：

假设总大小为 16，层级定义为 [2, 2, 4]。
1. Level 0 (size=2):
- u=1 (前面无层级), s=2, l=8 (后面 2*4)。
- 变换逻辑：(l s u) -> (l u) s 即 (8 2 1) -> (8 1) 2。
- 效果：将连续的 2 个元素归为一组。
- 结果：[[0,1], [2,3], [4,5], ...] （这是最细粒度的节点内分组）。

2. Level 1 (size=2):
- u=2, s=2, l=4。
- 变换逻辑：(4 2 2) -> (4 2) 2。
- 效果：每隔 2 个取一个，步长为 2。
- 结果：[[0,2], [1,3], [4,6], ...] （这是中间层分组）。

3. Level 2 (size=4):
- u=4, s=4, l=1。
- 变换逻辑：(1 4 4) -> (1 4) 4。
- 效果：每隔 4 个取一个，步长为 4。
- 结果：[[0,4,8,12], [1,5,9,13], ...] （这是最粗粒度的跨节点分组）。

总结：这个数学变换实际上是在做多维数组的转置与切片，它完美地将线性的 Rank 列表映射到了立体的层级拓扑结构中。

In [ ]:
try:
    import einops

    HAVE_EINOPS = True
except ImportError:
    HAVE_EINOPS = False
    
    

def create_hierarchical_groups(
    rank,
    ranks,
    hierarchical_group_sizes,
    create_gloo_process_groups=False,
    pg_options=None,
    timeout=None,
    group_desc=None,
):
    """Create hierarchical groups for a set of ranks.
    Taking a group size of 16 as example, so we have a total of 16 GPUs denoted by g0 ... g15.
    If the hierarchical group sizes are [2,2,4], we use 2 GPUs in the first and second level
    of sub-groups, and 4 GPUs in the last level of sub groups. The present function will
    create 8 level-1 sub-groups, 8 level-2 sub-groups and 4 level-3 sub-groups as:
        8 level-1 sub-groups:
            [g0, g1], [g2, g3], [g4, g5], [g6, g7], [g8, g9], [g10, g11], [g12, g13], [g14, g15]
        8 level-2 sub-groups:
            [g0, g2], [g1, g3], [g4, g6], [g5, g7], [g8, g10], [g9, g11], [g12, g14], [g13, g15]
        4 level-3 sub-groups:
            [g0, g4, g8, g12], [g1, g5, g9, g13], [g2, g6, g10, g14], [g3, g7, g11, g15]
    """

    if not HAVE_EINOPS:
        raise ImportError("einops is not installed. Please install it with `pip install einops`.")

    hierarchical_groups = []
    hierarchical_groups_gloo = []
    if not isinstance(pg_options, list):
        pg_options = [pg_options] * len(hierarchical_group_sizes)
    for level in range(len(hierarchical_group_sizes)):
        rearranged_ranks = einops.rearrange(
            np.array(ranks),
            "(l s u) -> (l u) s",
            u=int(np.prod(hierarchical_group_sizes[:level])),      # 前面的层级累积大小
            s=hierarchical_group_sizes[level],                     # 当前层级的大小
            l=int(np.prod(hierarchical_group_sizes[level + 1 :])), # 后面的层级累积大小
        ).tolist()
        
        '''
        创建通信组:
                 遍历生成的 rearranged_ranks，对每一个子列表调用 create_group。
                 - NCCL Group: 用于 GPU 之间的高速数据传输（默认）。
                 - Gloo Group: 如果 create_gloo_process_groups=True，还会创建一个 CPU 层面的通信组。
                               这通常用于调试、元数据同步或当 NCCL 不可用时作为降级方案
        '''
        for sub_ranks in rearranged_ranks:
            '''NCCL Group'''
            sub_group = create_group(
                sub_ranks,
                timeout=timeout,
                pg_options=pg_options[level],
                group_desc=f"HIERARCHICAL_{group_desc}_L{level}",
            )
            
            '''Gloo Group'''
            if create_gloo_process_groups:
                sub_group_gloo = create_group(
                    sub_ranks,
                    timeout=timeout,
                    backend="gloo",
                    pg_options=pg_options[level],
                    group_desc=f"HIERARCHICAL_{group_desc}_GLOO_L{level}",
                )
            else:
                sub_group_gloo = None
                
            if rank in sub_ranks:
                hierarchical_groups.append(sub_group)
                hierarchical_groups_gloo.append(sub_group_gloo)
                
    assert rank not in ranks or len(hierarchical_groups) == len(hierarchical_group_sizes)
    assert rank not in ranks or len(hierarchical_groups_gloo) == len(hierarchical_group_sizes)
    return hierarchical_groups, hierarchical_groups_gloo

## 创建通信组:
遍历生成的 rearranged_ranks，<font color='red'>对每一个子列表调用 create_group。</font>
- NCCL Group: 用于 GPU 之间的高速数据传输（默认）。 
- Gloo Group: 如果 create_gloo_process_groups=True，还会创建一个 CPU 层面的通信组。
              这通常用于调试、元数据同步或当 NCCL 不可用时作为降级方案
              


## 为什么要这样做？（实际价值）
这段代码通常配合 Ring Attention 或 Sequence Parallelism 使用。
1. 优化 Ring AllReduce：在标准的 Ring AllReduce 中，数据是传给下一个邻居的。如果使用分层组，我们可以先在 L0 层（NVLink）做完局部聚合，再在 L2 层（IB）做长距离传输。这比直接在 IB 上做 16 卡的 Ring 要快得多。
2. Overlap 计算与通信：不同的层级可以并发执行。例如，L0 层的通信可以和 L1 层的计算重叠。
3. 拓扑感知：通过这种方式，软件层面的通信组严格对齐了硬件层面的物理连线，避免了“舍近求远”的低效通信。

# 举例说明
- 假设有 16 张卡，层级配置为 [2, 2, 4]。

| 层级 | 组大小 | 分组逻辑 | 典型物理含义 | 示例组 |
| :--- | :--- | :--- | :--- | :--- |
| L0 | 2 | 连续相邻 | 同一个 NVSwitch 下的 GPU | `[g0, g1]`, `[g2, g3]`... |
| L1 | 2 | 间隔为 2 | 同一个节点内的不同 NVSwitch | `[g0, g2]`, `[g1, g3]`... |
| L2 | 4 | 间隔为 4 | 跨节点互联 | `[g0, g4, g8, g12]`... |


### 1. <font color='green'>一个 GPU 会同时属于多个小组</font>- <font color='red'>通信只发生在同组之间（同组属于同一个职责等级）</font>

 <font color='blue'>g0 并不是只在 L0 里和 g1 玩，在 L1 里又只和 g2 玩。实际上，g0 同时拥有多重身份：</font>
- 在 L0 层级，g0 的搭档是 g1。
- 在 L1 层级，g0 的搭档变成了 g2。
- 在 L2 层级，g0 的搭档是 g4、g8、g12。
在分布式训练的实际通信中，g0 会根据当前需要执行的通信原语（比如是节点内聚合还是跨节点同步），自动选择对应层级的小组进行数据交换。

### 2. 为什么 L1 是 [g0, g2] 而不是 [g0, 1, 2, 3]？
- 你可能会好奇，既然 L0 的 [g0, g1] 是连续相邻的，为什么 L1 的分组不是把 L0 的两个小组直接合并成 [g0, g1, g2, g3]，而是变成了“隔空取物”的 [g0, g2] 和 [g1, g3]？

 <font color='green'>这其实是为了避免通信冲突，实现并行传输：</font>
- 如果 L1 分组是 [g0, g1, g2, g3]，那么 g0 和 g1 之间既要在 L0 通信，又要在 L1 通信，容易造成信道拥堵。
- 采用 [g0, g2] 和 [g1, g3] 这种交叉分组后：
    - g0 和 g2 可以在 L1 层级同时通信。
    - g1 和 g3 也可以在 L1 层级同时通信。
    - 这两组通信互不干扰，相当于把原本串行的任务变成了并行处理，极大地提升了节点内跨 NVSwitch 的带宽利用率。
    
简单来说，这种看似复杂的数学切分（einops 变换），本质上是在给硬件的通信链路“规划最优交通路线”，确保数据能最快、最不堵车地到达目的地。